# Fine-Tuning BERT for Chunking (Token Classification)

## Objective
The objective of this project is to fine-tune a transformer model (DistilBERT) to perform **Chunking (Phrase Detection)** using token classification.

## Dataset
We use the **CoNLL-2003 dataset**, which contains:
- Tokens (words)
- POS tags
- Chunk tags (B-NP, I-NP, etc.)
- Named Entity tags

In this project, we focus on **Chunk Tags**.

## Pipeline
Raw Data → Tokenization → Label Alignment → Model → Training → Evaluation → Inference → Comparison

# Install Libraries

In [24]:
!pip install transformers datasets seqeval -q

# Import Libraries

In [25]:
import pandas as pd
import numpy as np

from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification
from datasets import Dataset, DatasetDict
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score

# Load Dataset

In [26]:
from google.colab import files

# Upload files manually
uploaded = files.upload()

# After upload, filenames will be available
train_file = "train.txt"
val_file = "valid.txt"
test_file = "test.txt"


def read_conll_file(filepath):
    sentences = []
    labels = []

    words = []
    chunk_tags = []

    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip() == "":
                if words:
                    sentences.append(words)
                    labels.append(chunk_tags)
                    words = []
                    chunk_tags = []
            else:
                parts = line.strip().split()

                # Skip DOCSTART lines
                if parts[0] == "-DOCSTART-":
                    continue

                if len(parts) >= 3:
                    words.append(parts[0])
                    chunk_tags.append(parts[2])  # Chunk column

    return sentences, labels


train_texts, train_labels = read_conll_file(train_file)
val_texts, val_labels = read_conll_file(val_file)
test_texts, test_labels = read_conll_file(test_file)

print("Sample Sentence:", train_texts[0])
print("Sample Labels:", train_labels[0])

Saving test.txt to test (2).txt
Saving train.txt to train (2).txt
Saving valid.txt to valid (2).txt
Sample Sentence: ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.']
Sample Labels: ['B-NP', 'B-VP', 'B-NP', 'I-NP', 'B-VP', 'I-VP', 'B-NP', 'I-NP', 'O']


In [27]:
train_dataset = Dataset.from_dict({
    "tokens": train_texts,
    "labels": train_labels
})

val_dataset = Dataset.from_dict({
    "tokens": val_texts,
    "labels": val_labels
})

test_dataset = Dataset.from_dict({
    "tokens": test_texts,
    "labels": test_labels
})

dataset = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset,
    "test": test_dataset
})

dataset

DatasetDict({
    train: Dataset({
        features: ['tokens', 'labels'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['tokens', 'labels'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['tokens', 'labels'],
        num_rows: 3453
    })
})

# Label Mapping

In [28]:
all_labels = []
for split in ["train", "validation", "test"]:
    for doc_labels in dataset[split]["labels"]:
        all_labels.extend(doc_labels)

label_list = sorted(list(set(all_labels)))

label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}

print(label_list)

['B-ADJP', 'B-ADVP', 'B-CONJP', 'B-INTJ', 'B-LST', 'B-NP', 'B-PP', 'B-PRT', 'B-SBAR', 'B-VP', 'I-ADJP', 'I-ADVP', 'I-CONJP', 'I-INTJ', 'I-LST', 'I-NP', 'I-PP', 'I-PRT', 'I-SBAR', 'I-VP', 'O']


# Tokenizer

In [29]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# Tokenization + Label Alignment

In [ ]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []

    for i, label in enumerate(examples["labels"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label2id[label[word_idx]])
            else:
                label_ids.append(-100) # Assign -100 for subsequent subword tokens

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)

# Model Setup

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

# Training Arguments

In [32]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="no",
    logging_dir="./logs"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


# Metrics

In [33]:
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [id2label[p] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    true_labels = [
        [id2label[l] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    return {
        "precision": precision_score(true_labels, true_predictions),
        "recall": recall_score(true_labels, true_predictions),
        "f1": f1_score(true_labels, true_predictions)
    }

# Trainer

In [34]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# Train Model

In [35]:
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.237164,0.214175,0.910860,0.899680,0.905235
2,0.170305,0.199722,0.914991,0.908401,0.911684


TrainOutput(global_step=3512, training_loss=0.25268519410104034, metrics={'train_runtime': 186.8405, 'train_samples_per_second': 150.299, 'train_steps_per_second': 18.797, 'total_flos': 296775812486196.0, 'train_loss': 0.25268519410104034, 'epoch': 2.0})

# Evaluation

In [36]:
trainer.evaluate()

{'eval_loss': 0.1997224986553192,
 'eval_precision': 0.9149912160843255,
 'eval_recall': 0.9084008914440748,
 'eval_f1': 0.9116841439893677,
 'eval_runtime': 5.6694,
 'eval_samples_per_second': 573.255,
 'eval_steps_per_second': 71.789,
 'epoch': 2.0}

# Inference

In [39]:
sentence = "John works at Google in California"

tokens = sentence.split()

tokenizer_output = tokenizer(tokens, return_tensors="pt", is_split_into_words=True)
# Move inputs to the same device as the model
inputs = {k: v.to(model.device) for k, v in tokenizer_output.items()}

outputs = model(**inputs).logits
predictions = np.argmax(outputs.detach().cpu().numpy(), axis=2)

word_ids = tokenizer_output.word_ids()
predicted_labels = []

for idx, word_id in enumerate(word_ids):
    if word_id is not None:
        predicted_labels.append(id2label[predictions[0][idx]])

print(list(zip(tokens, predicted_labels)))

[('John', 'B-NP'), ('works', 'B-VP'), ('at', 'B-PP'), ('Google', 'B-NP'), ('in', 'B-PP'), ('California', 'B-NP')]


# Comparison

## Comparison: POS Tagging vs Chunking

- POS Tagging identifies the grammatical role of each word (e.g., noun, verb).
- Chunking groups words into meaningful phrases (e.g., noun phrase, verb phrase).

### Key Difference:
- POS Tagging → Word-level classification (simpler)
- Chunking → Phrase-level grouping (more complex)

### Difficulty:
- POS Tagging: Easy
- Chunking: Medium

### Observation:
Chunking provides better structural understanding of sentences compared to POS tagging.

# Conclusion

## Conclusion

In this project, we successfully fine-tuned a DistilBERT model for chunking using token classification.

### Key Achievements:
- Performed tokenization and label alignment
- Trained transformer model
- Evaluated using seqeval metrics
- Performed inference on custom sentence

### Challenges:
- Handling subword tokenization
- Aligning labels correctly

### Final Insight:
Transformer models like BERT are highly effective for sequence labeling tasks such as chunking.